In [0]:
%sql
-- 1. Створюємо таблицю
CREATE OR REPLACE TABLE crude_ops.bronze.sap_operations (
    well_id STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    op STRING,
    phase STRING
);

-- 2. Заливаємо "м'ясо" (насичені дані на 2019-12-04)
INSERT INTO crude_ops.bronze.sap_operations VALUES
('31/5-7 Eos', '2019-12-03 23:00:00', '2019-12-04 02:00:00', 'Preparing Equipment', 'Phase 2'),
('31/5-7 Eos', '2019-12-04 02:00:00', '2019-12-04 04:30:00', 'Tripping In', 'Phase 2'),
('31/5-7 Eos', '2019-12-04 04:30:00', '2019-12-04 08:00:00', 'Drilling', 'Phase 2'),
('31/5-7 Eos', '2019-12-04 08:00:00', '2019-12-04 09:30:00', 'Connection', 'Phase 2'),
('31/5-7 Eos', '2019-12-04 09:30:00', '2019-12-04 12:00:00', 'Drilling', 'Phase 2'),
('31/5-7 Eos', '2019-12-04 12:00:00', '2019-12-04 14:00:00', 'Circulating', 'Phase 2'),
('31/5-7 Eos', '2019-12-04 14:00:00', '2019-12-04 17:00:00', 'Tripping Out', 'Phase 2'),
('31/5-7 Eos', '2019-12-04 17:00:00', '2019-12-04 19:00:00', 'Casing & Cementing', 'Phase 3 Transition'),
('31/5-7 Eos', '2019-12-04 19:00:00', '2019-12-04 22:00:00', 'Drilling', 'Phase 3'),
('31/5-7 Eos', '2019-12-04 22:00:00', '2019-12-05 02:00:00', 'Surveying', 'Phase 3');

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

schema = StructType([
    StructField("well_id", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("op", StringType(), True),
    StructField("phase", StringType(), True)
])

# Генеруємо насичений графік на добу (2019-12-04)
# Тепер у нас з'являється Phase 2 та Phase 3
data = [
    ("31/5-7 Eos", "2019-12-03 23:00:00", "2019-12-04 02:00:00", "Preparing Equipment", "Phase 2"),
    ("31/5-7 Eos", "2019-12-04 02:00:00", "2019-12-04 04:30:00", "Tripping In", "Phase 2"),
    ("31/5-7 Eos", "2019-12-04 04:30:00", "2019-12-04 08:00:00", "Drilling", "Phase 2"),
    ("31/5-7 Eos", "2019-12-04 08:00:00", "2019-12-04 09:30:00", "Connection", "Phase 2"),
    ("31/5-7 Eos", "2019-12-04 09:30:00", "2019-12-04 12:00:00", "Drilling", "Phase 2"),
    ("31/5-7 Eos", "2019-12-04 12:00:00", "2019-12-04 14:00:00", "Circulating", "Phase 2"),
    ("31/5-7 Eos", "2019-12-04 14:00:00", "2019-12-04 17:00:00", "Tripping Out", "Phase 2"),
    ("31/5-7 Eos", "2019-12-04 17:00:00", "2019-12-04 19:00:00", "Casing & Cementing", "Phase 3 Transition"),
    ("31/5-7 Eos", "2019-12-04 19:00:00", "2019-12-04 22:00:00", "Drilling", "Phase 3"),
    ("31/5-7 Eos", "2019-12-04 22:00:00", "2019-12-05 02:00:00", "Surveying", "Phase 3")
]

sap_df = spark.createDataFrame(data, schema)
sap_df.write.format("delta").mode("overwrite").saveAsTable("crude_ops.bronze.sap_operations")

print("SAP table enriched with multiple operations and phases.")

In [0]:
# Scanning the LWD folder for time-series or high-resolution depth data
lwd_path = "/Volumes/equinor_asa_northern_lights/public/northernlights/31_5-7 Eos/05.LWD_Log_data/"

files = dbutils.fs.ls(lwd_path)
for f in files:
    # We are looking for larger files or specific keywords like 'Time', 'Log', 'Daily'
    print(f"File: {f.name} | Size: {f.size / 1024 / 1024:.2f} MB")

In [0]:
# Select one of the LAS files
las_file_path = "/Volumes/equinor_asa_northern_lights/public/northernlights/31_5-7 Eos/05.LWD_Log_data/WL_RAW_BHPR-GR-MECH_TIME_MWD_1.LAS"

# Read as text to see the structure and column names
raw_las_df = spark.read.text(las_file_path)

# Show the header
print("--- LAS File Header & Data Preview ---")
raw_las_df.limit(100).show(truncate=False)


In [0]:
# Attempting to read it as a delimited file if it looks like a table
try:
    # Often these files use multiple spaces as delimiters
    df_pressure = spark.read.option("header", "true").option("inferSchema", "true").csv(asc_file_path)
    
    print("Structure of the Pressure Data:")
    df_pressure.printSchema()
    
    print("Data Preview:")
    display(df_pressure.limit(10))
except Exception as e:
    print(f"Simple CSV read failed, likely due to complex header: {e}")